# Analysis of protein recruitment to MLOs

This notebook covers the following analysis steps:
- Read the *merge* image (2 channels), the cell ROIs (Cellpose, `.zip`), and the ilastik *Simple Segmentation* (`.tif`).
- Detect the MLOs, separate them, and measure intensity **inside** each MLO and in exterior **rings**, using exclusion zones of different sizes (clipped to the cell and excluding other MLOs).
- Provide **quality control (QC)** tools to check that MLOs are well segmented.
- Export `MLO_ring_results.csv` and `QC_MLO.csv`, compute the inside/outside ratio per MLO, and produce the summary figures.

**Notebook structure**
1. Install dependencies
2. Mount Google Drive
3. Configuration (paths and parameters)
4. Helper functions (I/O + measurement)
5. Calibrate erosion and exclusion distance from mean MLO size
6. Visualization helpers (display only — not used for measurements)
7. Select an image for QC
8. QC-B — Overlay image + MLO and cell contours
9. QC-C — Gallery of individual MLOs
10. QC-D — Shape/contrast metrics and flags
11. Gallery of MLOs to REVIEW
12. Cell-by-cell review (C1 | C2 | merge)
13. Per-cell control panels for all images (batch PNG export)
14. Measure one image (test)
15. Run the full batch and save the CSVs
16. Number of MLOs per cell
17. Inspect one image — all its cells
18. Log cells to discard
19. Apply the discard list to the results
20. Quick look at the results
21. Session info (reproducibility)
22. Compute the inside/outside ratio per MLO
23. Plot — inner signal vs erosion (per condition)
24. Plot — outer signal vs exclusion distance (per condition)
25. Plot — inside/outside ratio (all conditions)
26. Plot — inner vs outer signal, scatter (all conditions)

## 1. Install dependencies
Installs `roifile` and `imagecodecs` (not preinstalled in Colab).

In [ ]:
# Installs the packages this notebook needs
!pip install -q roifile imagecodecs
print("Done. If you hit an import error below, restart the session "
      "(Runtime -> Restart session) and re-run from here -- no need to reinstall.")

## 2. Mount Google Drive
Connects your Google Drive to read the images and save the results. Asks for authorization the first time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuration (**EDIT**)
Define the paths to folders (merge, Cellpose ROIs, ILASTIK segmentation, and output), the analysis parameters, and the condition/color scheme used later for the ratio and plots.

In [ ]:
# ---- PATHS (edit) ----
DIR_MERGE   = "/content/drive/MyDrive/MIE/Recruitment/Merge/"
DIR_CELLS   = "/content/drive/MyDrive/MIE/Recruitment/ROIs/"
DIR_ILASTIK = "/content/drive/MyDrive/MIE/Recruitment/Simple_Segmentation/"
DIR_OUT     = "/content/drive/MyDrive/MIE/Recruitment/Tables/"

EXT_IMG, EXT_ROI, EXT_SEG = ".tif", ".zip", ".tif"
SEG_SUFFIX = ""           # e.g. "_Simple Segmentation"

C1_INDEX, C2_INDEX = 0, 1 # which slice of the merge is C1 and C2 (0 = first)

# ---- MLO parameters (edit) ----
MIN_MLO_AREA = 4          # minimum area (px) -- like Analyze Particles > 4
MAX_MLO_AREA = 100000

# ---- rings: (name, inner_enlarge, outer_enlarge) in px
RINGS = [("Ring_1px_excl0", 0, 1)]

# ---- MLO core erosion
ERODE_LEVELS = [0]

# MLO class in ILASTIK: "auto" = smallest-area non-background class; or an integer (e.g. 2)
MLO_LABEL = "auto"

CLIP_TO_CELL       = True   # clip the ring to the containing cell
EXCLUDE_OTHER_MLOS = True   # exclude any other MLO from the ring

# ---- QC flag thresholds: threshold values to flag MLOs as abnormal ----
QC_AREA_MIN, QC_AREA_MAX = 6, 5000
QC_CIRC_MIN  = 0.30
QC_CONTRAST  = 1.10

# ---- per-MLO intensity filter used for the ratio + plots (sections 22-26) ----
READOUT    = "C1"
INTENS_MAX = 200         # an MLO is discarded if Max_<READOUT> reaches this in ANY region

# ---- conditions to be inferred from the filename ----
COND_LEVELS = ["SSR12"]   # format as follows: ["Condition_A", "Condition_B", ...]
COND_COLORS = {"SSR12": "#27AE60"}   # format as follows: {"Condition_A": "#Color_A", "Condition_B": "#Color_B", ...}

def condition_of(image_name): # type all conditions below
    if "SSR12"  in image_name: return "SSR12"
    if "Condition_B" in image_name: return "Condition_B"
    if "Condition_C" in image_name: return "Condition_C"
    return "Control"

import os
os.makedirs(DIR_OUT, exist_ok=True)
print("Config ready. Output in:", DIR_OUT)

## 4. Helper functions
I/O and measurement functions used by every section below.

In [ ]:
import os, glob, csv
import numpy as np
import tifffile, roifile
from scipy import ndimage as ndi
import matplotlib.pyplot as plt
from skimage.draw import polygon as sk_polygon
from skimage.segmentation import find_boundaries

def detect_mlo_label(seg):
    # Pixel value of the MLO class = smallest-area non-background class (background = most frequent)
    vals, counts = np.unique(seg, return_counts=True)
    background = vals[np.argmax(counts)]
    cand = sorted((c, v) for v, c in zip(vals, counts) if v != background and c > 0)
    return int(cand[0][1]) if cand else int(background) + 1

def load_merge(path):
    mg = np.asarray(tifffile.imread(path))
    if mg.ndim == 3 and mg.shape[0] <= 4:        # (C, Y, X)
        return mg[C1_INDEX], mg[C2_INDEX]
    if mg.ndim == 3 and mg.shape[-1] <= 4:       # (Y, X, C)
        return mg[..., C1_INDEX], mg[..., C2_INDEX]
    raise ValueError(f"Unrecognized merge shape: {mg.shape}")

def load_segmentation(path):
    seg = np.asarray(tifffile.imread(path))
    if seg.ndim > 2:
        seg = seg.reshape((-1,) + seg.shape[-2:])[0]
    return seg

def label_mlos(seg, mlo_val):
    mask = ndi.binary_fill_holes(seg == mlo_val)
    lab, n = ndi.label(mask)
    areas = ndi.sum(np.ones_like(lab), lab, index=np.arange(1, n + 1))
    keep = [i + 1 for i, a in enumerate(areas) if MIN_MLO_AREA <= a <= MAX_MLO_AREA]
    all_mlo = np.isin(lab, keep)
    return lab, keep, all_mlo

def rasterize_cells(zip_path, shape):
    lab = np.zeros(shape, dtype=np.int32)
    rois = roifile.roiread(zip_path)
    if not isinstance(rois, list):
        rois = [rois]
    for i, r in enumerate(rois, start=1):
        c = r.coordinates()
        if c is None or len(c) < 3:
            continue
        rr, cc = sk_polygon(c[:, 1], c[:, 0], shape=shape)
        lab[rr, cc] = i
    return lab, len(rois)

def region_stats(mask, c1, c2):
    n = int(mask.sum())
    if n == 0:
        out = {"Area": 0}
        for ch in ("C1", "C2"):
            for k in ("Mean", "StdDev", "Median", "Min", "Max", "IntDen"):
                out[f"{k}_{ch}"] = "NA"
        return out
    out = {"Area": n}
    for ch, img in (("C1", c1), ("C2", c2)):
        v = img[mask].astype(np.float64)
        out[f"Mean_{ch}"]   = round(float(v.mean()), 4)
        out[f"StdDev_{ch}"] = round(float(v.std()), 4)
        out[f"Median_{ch}"] = round(float(np.median(v)), 4)
        out[f"Min_{ch}"]    = float(v.min())
        out[f"Max_{ch}"]    = float(v.max())
        out[f"IntDen_{ch}"] = round(float(v.sum()), 4)
    return out

def perimeter(mask):
    from skimage.measure import perimeter as _p
    return float(_p(mask, neighborhood=8))

def mlos_by_cell(lab, keep, cell_lab):
    '''{cell_id: [mlo_index, ...]} using each MLO's centroid.'''
    mapping = {}
    for k, lbl in enumerate(keep, start=1):
        ys, xs = np.where(lab == lbl)
        cid = int(cell_lab[int(round(ys.mean())), int(round(xs.mean()))])
        mapping.setdefault(cid, []).append(k)
    return mapping

print("Functions loaded.")

## 5. Calibrate erosion and exclusion distance from mean MLO size (**EDIT**)
Scan **all** the Simple Segmentations, measure the area of each MLO, compute the **mean equivalent radius**, and set `ERODE_LEVELS` and `RINGS` as fractions of that radius.

Edit the fractions as desired. Check the printed summary before running the next code chunks.

In [ ]:
# ============================================================
# CALIBRATE parameters (erode / exclusion) from mean size
# ============================================================
# --- Fractions of the mean equivalent radius ---
ERODE_FRACS      = [0, 1/4, 1/2, 3/4, 1]                 # inner erosion
EXCLUSION_FRACS  = [0, 1/4, 1/3, 1/2, 3/4, 1, 3/2, 2]    # outer exclusion distance
RING_WIDTHS_PX   = [1]                                    # ring width (fixed px)

# Collect areas from all segmentations
areas = []
segs = sorted(glob.glob(os.path.join(DIR_ILASTIK, "*" + EXT_SEG)))
assert segs, f"No files found in {DIR_ILASTIK!r}. Check DIR_ILASTIK in the config."

for p_seg in segs:
    seg = np.asarray(tifffile.imread(p_seg))
    if seg.ndim > 2:
        seg = seg[..., 0]
    mv = detect_mlo_label(seg) if MLO_LABEL == "auto" else int(MLO_LABEL)
    mlo_mask = seg == mv
    labeled, n_lbl = ndi.label(mlo_mask)
    for lbl in range(1, n_lbl + 1):
        area = int((labeled == lbl).sum())
        if MIN_MLO_AREA <= area <= MAX_MLO_AREA:
            areas.append(area)

assert areas, "No MLOs found. Check MIN_MLO_AREA / MAX_MLO_AREA and DIR_ILASTIK."
areas = np.array(areas)

mean_area = float(np.mean(areas))
mean_r    = float(np.sqrt(mean_area / np.pi))   # equivalent radius

print(f"Images scanned       : {len(segs)}")
print(f"MLOs found            : {len(areas)}")
print(f"Mean area             : {mean_area:.1f} px²")
print(f"Mean equiv. radius    : {mean_r:.2f} px")
print()

def frac_to_px(frac, r):
    return int(round(frac * r))

# Erosion
seen_ep = set()
erode_px_ordered = []
for f in ERODE_FRACS:
    ep = frac_to_px(f, mean_r)
    if ep not in seen_ep:
        seen_ep.add(ep)
        erode_px_ordered.append(ep)
ERODE_LEVELS = erode_px_ordered

# Rings
RINGS = []
seen_combo = set()
for w in RING_WIDTHS_PX:
    for gf in EXCLUSION_FRACS:
        excl_px = frac_to_px(gf, mean_r)
        out_px = excl_px + w
        key    = (excl_px, out_px)
        if key in seen_combo:
            continue
        seen_combo.add(key)
        RINGS.append((f"Ring_{w}px_excl{excl_px}", excl_px, out_px))

print("━" * 60)
print(" Inner erosion  (erode = fraction × mean radius)")
for ep in ERODE_LEVELS:
    name = "Inner" if ep == 0 else f"Inner_erode{ep}"
    print(f"   {ep:>3} px   ->  {name}")
print()
print(" Outer rings  (exclusion distance = fraction × mean radius, fixed width)")
for name, excl_px, out_px in RINGS:
    print(f"   excl={excl_px:>3} px   ->  {name}")
print("━" * 60)
print(f"\nTotal: {len(ERODE_LEVELS)} erosion levels | {len(RINGS)} rings")

# ============================================================
# Export calibration table (used for readable axis labels below)
# ============================================================
import pandas as pd

FRAC_STR_E = {0: "0r", 1/4: "1/4r", 1/2: "1/2r", 3/4: "3/4r", 1.0: "1r"}
FRAC_STR_G = {0: "0r", 1/4: "1/4r", 1/3: "1/3r", 1/2: "1/2r",
              3/4: "3/4r", 1.0: "1r", 3/2: "3/2r", 2.0: "2r"}

calib_rows = [{"Region": "__mean_r__", "type": "metadata",
               "fraction": "mean_r", "px_value": round(mean_r, 3),
               "ring_w": None, "mean_r": round(mean_r, 3),
               "axis_label": f"r={mean_r:.2f} px  (mean area={mean_area:.1f} px2)"}]

seen_ep3 = set()
for f in ERODE_FRACS:
    px = frac_to_px(f, mean_r)
    if px in seen_ep3: continue
    seen_ep3.add(px)
    name  = "Inner" if px == 0 else f"Inner_erode{px}"
    fstr  = FRAC_STR_E.get(f, f"{f:.3g}r")
    label = "0\n(0r)" if px == 0 else f"{px}px\n({fstr})"
    calib_rows.append({"Region": name, "type": "interior", "fraction": fstr, "px_value": px,
                       "ring_w": None, "mean_r": round(mean_r, 3), "axis_label": label})

seen_gp3 = set()
for gf in EXCLUSION_FRACS:
    gp = frac_to_px(gf, mean_r)
    if gp in seen_gp3: continue
    seen_gp3.add(gp)
    gstr = FRAC_STR_G.get(gf, f"{gf:.3g}r")
    for w in RING_WIDTHS_PX:
        name  = f"Ring_{w}px_excl{gp}"
        label = f"{gp}px\n({gstr})" if gp > 0 else "0\n(0r)"
        calib_rows.append({"Region": name, "type": "ring", "fraction": gstr, "px_value": gp,
                           "ring_w": w, "mean_r": round(mean_r, 3), "axis_label": label})

df_calib = pd.DataFrame(calib_rows)
calib_path = os.path.join(DIR_OUT, "MLO_calibration.csv")
df_calib.to_csv(calib_path, index=False)
print(f"\nCalibration saved: {calib_path}")

## 6. Visualization helpers
Display-only helpers (normalization for viewing, contour overlays, cropping, per-cell panel) shared by every QC and review section below.

In [ ]:
def normalize_img(x, gamma=0.7):
    '''Percentile-normalize + gamma boost for DISPLAY ONLY (not used for measurements).'''
    x = x.astype(np.float64)
    lo, hi = np.percentile(x, 1), np.percentile(x, 99.5)
    return np.clip((x - lo) / (hi - lo + 1e-9), 0, 1) ** gamma

def bbox_slice(mask, pad, shape):
    '''Bounding box of a mask, padded, clipped to the image shape.'''
    ys, xs = np.where(mask)
    y0, y1 = max(0, ys.min() - pad), min(shape[0], ys.max() + pad + 1)
    x0, x1 = max(0, xs.min() - pad), min(shape[1], xs.max() + pad + 1)
    return slice(y0, y1), slice(x0, x1)

def with_contours(gray, mlo_mask, cell_mask, darken_outside=False):
    '''Grayscale image -> RGB with MLO contour (cyan) and cell contour (yellow).'''
    rgb = np.dstack([gray, gray, gray])
    if darken_outside:
        rgb[~cell_mask] = rgb[~cell_mask] * 0.25
    rgb[find_boundaries(cell_mask, mode="outer")] = [1, 1, 0]
    rgb[find_boundaries(mlo_mask,  mode="outer")] = [0, 1, 1]
    return np.clip(rgb, 0, 1)

def merge_rgb(c1_norm, c2_norm, cell_mask=None, darken_outside=False):
    '''C1 green / C2 magenta merge (no contours yet).'''
    mg = np.dstack([c2_norm, c1_norm, c2_norm])
    if darken_outside and cell_mask is not None:
        mg[~cell_mask] = mg[~cell_mask] * 0.25
    return mg

def view_cell(cid, c1, c2, lab, cell_lab, all_mlo, pad=12, gamma=0.7,
              darken_outside=True, title=None, mlo_ids=None):
    '''Show C1 | C2 | merge for one cell, MLOs in cyan, cell border in yellow.'''
    if not (cell_lab == cid).any():
        print(f"Cell {cid} has no pixels."); return
    sl = bbox_slice(cell_lab == cid, pad, cell_lab.shape)
    c1c, c2c = normalize_img(c1[sl], gamma), normalize_img(c2[sl], gamma)
    cellm = (cell_lab[sl] == cid)
    mlos  = all_mlo[sl] & cellm

    mg = merge_rgb(c1c, c2c, cellm, darken_outside)
    mg[find_boundaries(cellm, mode="outer")] = [1, 1, 0]
    mg[find_boundaries(mlos,  mode="outer")] = [0, 1, 1]

    fig, ax = plt.subplots(1, 3, figsize=(15, 5.2))
    ax[0].imshow(with_contours(c1c, mlos, cellm)); ax[0].set_title("C1 + MLOs")
    ax[1].imshow(with_contours(c2c, mlos, cellm)); ax[1].set_title("C2 + MLOs")
    ax[2].imshow(np.clip(mg, 0, 1));               ax[2].set_title("Merge (C1 green / C2 magenta)")
    for a in ax: a.axis("off")
    default_title = f"Cell {cid}" + (f" — {len(mlo_ids)} MLOs" if mlo_ids is not None else "")
    plt.suptitle(title or default_title, y=1.02)
    plt.tight_layout(); plt.show()
    if mlo_ids is not None:
        print(f"Cell {cid}: MLOs =", [f"MLO_{k:04d}" for k in mlo_ids])

print("Visualization helpers ready.")

## 7. Select an image for QC
Load everything needed for the QC sections below (segmentation, MLOs, merge, cells) for ONE image, and print the segmentation values to confirm `MLO_LABEL` (section 3) picked the right class.

> Defines `base`, `seg`, `mlo_val`, `lab`, `keep`, `all_mlo`, `c1`, `c2`, `cell_lab` — used by sections 8-12.

In [ ]:
IMG_IDX = 0   # change the index to check a different image

imgs = sorted(glob.glob(os.path.join(DIR_MERGE, "*" + EXT_IMG)))
assert imgs, f"No images found in {DIR_MERGE}"
base    = os.path.splitext(os.path.basename(imgs[IMG_IDX]))[0]
p_merge = imgs[IMG_IDX]
p_cell  = os.path.join(DIR_CELLS, base + EXT_ROI)
p_seg   = os.path.join(DIR_ILASTIK, base + SEG_SUFFIX + EXT_SEG)
print("Selected image:", base)
print("  merge  :", p_merge, "OK" if os.path.exists(p_merge) else "MISSING")
print("  cell   :", p_cell,  "OK" if os.path.exists(p_cell)  else "MISSING")
print("  ilastik:", p_seg,   "OK" if os.path.exists(p_seg)   else "MISSING")

seg = load_segmentation(p_seg)
vals, counts = np.unique(seg, return_counts=True)
print("\nValues in the segmentation (value -> nº of pixels):")
for v, c in zip(vals, counts): print(f"   {v} -> {c}")

mlo_val = detect_mlo_label(seg) if MLO_LABEL == "auto" else int(MLO_LABEL)
print("\n>>> MLO class used =", mlo_val)

lab, keep, all_mlo = label_mlos(seg, mlo_val)
c1, c2 = load_merge(p_merge)
cell_lab, n_cells = rasterize_cells(p_cell, c1.shape)

print(f"Total MLOs detected: {lab.max()}  |  after area filter (>={MIN_MLO_AREA}px): {len(keep)}")
print(f"Cells detected: {n_cells}")

## 8. QC-B — Overlay: image + MLO and cell contours
Draws the image (C1 green / C2 magenta) with the MLO contours (cyan) and cell contours (yellow). Use `ZOOM` to look at a region in detail.

> Requires having run section 7.

In [ ]:
BORDER_THICKNESS = 2     # MLO contour thickness in px
GAMMA_QC         = 0.7   # <1 brightens dim signals
SHOW_CELLS       = True

g = normalize_img(c1, GAMMA_QC)   # channel 1 -> green
m = normalize_img(c2, GAMMA_QC)   # channel 2 -> magenta

rgb = np.zeros(c1.shape + (3,))
rgb[..., 0] = m; rgb[..., 1] = g; rgb[..., 2] = m

b_mlo = find_boundaries(all_mlo, mode="outer")
if BORDER_THICKNESS > 1:
    b_mlo = ndi.binary_dilation(b_mlo, iterations=BORDER_THICKNESS - 1)
rgb[b_mlo] = [0, 1, 1]

if SHOW_CELLS:
    b_cell = find_boundaries(cell_lab, mode="thick")
    if BORDER_THICKNESS > 1:
        b_cell = ndi.binary_dilation(b_cell, iterations=BORDER_THICKNESS - 1)
    rgb[b_cell] = [1, 1, 0]

ZOOM = None   # or e.g. (0, 500, 0, 500) -> (y0, y1, x0, x1)
view = rgb if ZOOM is None else rgb[ZOOM[0]:ZOOM[1], ZOOM[2]:ZOOM[3]]
plt.figure(figsize=(12, 12)); plt.imshow(np.clip(view, 0, 1)); plt.axis("off")
plt.title(f"{base} — C1 green / C2 magenta — MLOs (cyan), cells (yellow)")
plt.show()

## 9. QC-C — Gallery of individual MLOs
Shows a mosaic with crops of the first `N_SHOW` MLOs and their contour.

> Requires having run section 7.

In [ ]:
N_SHOW = 24          # how many MLOs to show (increase carefully)
VIS_CHANNEL = c2     # channel to visualize (c1 or c2)

sel = keep[:N_SHOW]
ncol = 6; nrow = int(np.ceil(len(sel) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 2.2, nrow * 2.2))
axes = np.atleast_2d(axes)
for ax in axes.ravel(): ax.axis("off")
for j, lbl in enumerate(sel):
    sl = bbox_slice(lab == lbl, 6, VIS_CHANNEL.shape)
    crop = normalize_img(VIS_CHANNEL[sl])
    mloc = (lab[sl] == lbl)
    rgb_c = np.dstack([crop] * 3)
    rgb_c[find_boundaries(mloc, mode="outer")] = [0, 1, 1]
    ax = axes.ravel()[j]
    ax.imshow(rgb_c); ax.set_title(f"MLO_{keep.index(lbl)+1:04d}\n{int(mloc.sum())}px", fontsize=8)
plt.suptitle(f"{base} — first {len(sel)} MLOs", y=1.02); plt.tight_layout(); plt.show()

## 10. QC-D — Shape/contrast metrics and flags
Computes, per MLO, area, circularity, and inner/edge contrast, plus a **flag** (`REVIEW`/`ok`) based on the thresholds from section 3.

> Requires section 7; creates the `qc` table used by section 11.

In [ ]:
def qc_single_image(lab, keep, all_mlo, c1, c2, cell_lab):
    rows = []
    pad = 5
    for k, lbl in enumerate(keep, start=1):
        ys, xs = np.where(lab == lbl)
        cy, cx = int(round(ys.mean())), int(round(xs.mean()))
        cell_id = int(cell_lab[cy, cx])
        sl = bbox_slice(lab == lbl, pad, c1.shape)
        m = (lab[sl] == lbl); others = all_mlo[sl] & ~m
        dist = ndi.distance_transform_edt(~m)
        area = int(m.sum()); per = perimeter(m)
        circ = min(4 * np.pi * area / per**2, 1.0) if per > 0 else 0.0
        bq = (dist > 0) & (dist <= 2) & (~others)
        m1i = float(c1[sl][m].mean()); m2i = float(c2[sl][m].mean())
        if bq.sum() > 0:
            m1b = float(c1[sl][bq].mean()); m2b = float(c2[sl][bq].mean())
            ct1 = m1i / m1b if m1b else np.nan; ct2 = m2i / m2b if m2b else np.nan
        else:
            m1b = m2b = ct1 = ct2 = np.nan
        reasons = []
        if area < QC_AREA_MIN: reasons.append("small_area")
        if area > QC_AREA_MAX: reasons.append("large_area")
        if circ < QC_CIRC_MIN: reasons.append("irregular_shape")
        if not np.isnan(ct1) and ct1 < QC_CONTRAST: reasons.append("low_contrast_C1")
        if cell_id == 0: reasons.append("outside_cell")
        rows.append(dict(MLO_ID=f"MLO_{k:04d}", Cell_ID=cell_id, Area=area,
                          Circ=round(circ, 3), Contrast_C1=round(ct1, 3) if not np.isnan(ct1) else np.nan,
                          Contrast_C2=round(ct2, 3) if not np.isnan(ct2) else np.nan,
                          Flag="REVIEW" if reasons else "ok", Reasons=";".join(reasons)))
    return pd.DataFrame(rows)

qc = qc_single_image(lab, keep, all_mlo, c1, c2, cell_lab)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ax[0].hist(qc.Area, bins=30); ax[0].set_title("Area (px)")
ax[1].hist(qc.Circ, bins=30); ax[1].set_title("Circularity")
ax[2].hist(qc.Contrast_C1.dropna(), bins=30); ax[2].set_title("Contrast C1 (inner/edge)")
plt.tight_layout(); plt.show()

print("Total MLOs:", len(qc), "| ok:", (qc.Flag == "ok").sum(), "| REVIEW:", (qc.Flag == "REVIEW").sum())
reasons_all = qc.loc[qc.Reasons != "", "Reasons"].str.split(";").explode()
print("\nReview reasons:"); print(reasons_all.value_counts())
print("\nMLOs flagged for review:")
qc[qc.Flag == "REVIEW"]

## 11. Gallery of MLOs to REVIEW
Shows only the MLOs flagged as `REVIEW` by QC-D, each with its reason.

> Requires section 10, which creates the `qc` table.

In [ ]:
VIS_CHANNEL = c1
PAD = 8
NCOL = 6

flagged = qc[qc.Flag == "REVIEW"].reset_index(drop=True)
print(f"MLOs flagged for review: {len(flagged)} of {len(qc)}")

if len(flagged) == 0:
    print("No flagged MLOs. ✔")
else:
    n = len(flagged); nrow = int(np.ceil(n / NCOL))
    fig, axes = plt.subplots(nrow, NCOL, figsize=(NCOL * 2.4, nrow * 2.6))
    axes = np.atleast_2d(axes)
    for ax in axes.ravel(): ax.axis("off")
    for j, row in flagged.iterrows():
        k = int(row.MLO_ID.split("_")[1]) - 1
        lbl = keep[k]
        sl = bbox_slice(lab == lbl, PAD, VIS_CHANNEL.shape)
        crop = normalize_img(VIS_CHANNEL[sl])
        mloc = (lab[sl] == lbl)
        rgb = np.dstack([crop * 0, crop, crop * 0])
        rgb[find_boundaries(mloc, mode="outer")] = [0, 1, 1]
        ax = axes.ravel()[j]
        ax.imshow(np.clip(rgb, 0, 1))
        ax.set_title(f"{row.MLO_ID}\n{row.Area}px  circ={row.Circ}\n{row.Reasons}", fontsize=7)
    plt.suptitle(f"{base} — MLOs to REVIEW", y=1.01)
    plt.tight_layout(); plt.show()
    flagged[["MLO_ID", "Cell_ID", "Area", "Circ", "Contrast_C1", "Reasons"]]

## 12. Cell-by-cell review (C1 | C2 | merge)
Goes through the cells of the current image (section 7) and shows, per cell, both channels and the merge with the MLOs marked, using the shared `view_cell` helper (section 6).

> Requires section 7.

In [ ]:
VIEW    = "all"   # "single" = just one (CELL_ID); "all" = go through all
CELL_ID = 10

mlo_map = mlos_by_cell(lab, keep, cell_lab)

if VIEW == "single":
    view_cell(CELL_ID, c1, c2, lab, cell_lab, all_mlo, mlo_ids=mlo_map.get(CELL_ID, []))
else:
    for cid in range(1, cell_lab.max() + 1):
        view_cell(cid, c1, c2, lab, cell_lab, all_mlo, mlo_ids=mlo_map.get(cid, []))

## 13. Per-cell control panels for all images (batch PNG export)
Generates one PNG per cell (C1 | C2 | merge, MLOs marked) for every image in `DIR_MERGE`, using the same shared functions and paths from sections 3, 4, and 6.

In [ ]:
CELL_PANEL_DIR = os.path.join(DIR_OUT, "cell_panels")
os.makedirs(CELL_PANEL_DIR, exist_ok=True)
ONLY_IMAGES = None   # None = all; or a list e.g. ["hSmaug1_C1_01"]

imgs_all = sorted(glob.glob(os.path.join(DIR_MERGE, "*" + EXT_IMG)))
n_panels = 0
for p in imgs_all:
    base_p = os.path.splitext(os.path.basename(p))[0]
    if ONLY_IMAGES is not None and base_p not in ONLY_IMAGES:
        continue
    pc = os.path.join(DIR_CELLS, base_p + EXT_ROI)
    ps = os.path.join(DIR_ILASTIK, base_p + SEG_SUFFIX + EXT_SEG)
    if not (os.path.exists(pc) and os.path.exists(ps)):
        print("[skip]", base_p, "(missing .zip or .tif)"); continue

    seg_p = load_segmentation(ps)
    mlo_val_p = detect_mlo_label(seg_p) if MLO_LABEL == "auto" else int(MLO_LABEL)
    lab_p, keep_p, all_mlo_p = label_mlos(seg_p, mlo_val_p)
    c1_p, c2_p = load_merge(p)
    cell_lab_p, _ = rasterize_cells(pc, c1_p.shape)

    sub_out = os.path.join(CELL_PANEL_DIR, base_p); os.makedirs(sub_out, exist_ok=True)
    for cid in range(1, cell_lab_p.max() + 1):
        if not (cell_lab_p == cid).any():
            continue
        sl = bbox_slice(cell_lab_p == cid, 10, cell_lab_p.shape)
        c1c, c2c = normalize_img(c1_p[sl]), normalize_img(c2_p[sl])
        cellm = (cell_lab_p[sl] == cid); mlos = all_mlo_p[sl] & cellm
        mg = merge_rgb(c1c, c2c, cellm, darken_outside=True)
        mg[find_boundaries(cellm, mode="outer")] = [1, 1, 0]
        mg[find_boundaries(mlos,  mode="outer")] = [0, 1, 1]
        n_mlos = len(np.unique(lab_p[sl][mlos]))

        fig, ax = plt.subplots(1, 3, figsize=(13, 4.6))
        ax[0].imshow(with_contours(c1c, mlos, cellm, darken_outside=True)); ax[0].set_title("C1")
        ax[1].imshow(with_contours(c2c, mlos, cellm, darken_outside=True)); ax[1].set_title("C2")
        ax[2].imshow(np.clip(mg, 0, 1)); ax[2].set_title("Merge (C1 green / C2 magenta)")
        for a in ax: a.axis("off")
        fig.suptitle(f"{base_p} | cell {cid} | {n_mlos} MLOs", fontsize=12, y=1.02)
        fig.savefig(os.path.join(sub_out, f"cell_{cid:02d}.png"), dpi=130, bbox_inches="tight")
        plt.close(fig); n_panels += 1
    print(f"[ok] {base_p}: {cell_lab_p.max()} cells")

print(f"\nDone. {n_panels} panels (C1|C2|merge) in: {CELL_PANEL_DIR}")

## 14. Measure one image (test)
Measures the inner region + rings for the selected image (section 7) and shows the first rows.

In [ ]:
def measure_image(base, c1, c2, lab, keep, all_mlo, cell_lab):
    rows = []
    max_outer = max(o for _, _, o in RINGS); pad = max_outer + 2
    H, W = c1.shape
    for k, lbl in enumerate(keep, start=1):
        mlo_id = f"MLO_{k:04d}"
        ys, xs = np.where(lab == lbl)
        cy, cx = int(round(ys.mean())), int(round(xs.mean()))
        cell_id = int(cell_lab[cy, cx])
        sl = bbox_slice(lab == lbl, pad, (H, W))
        m = (lab[sl] == lbl); c1l = c1[sl]; c2l = c2[sl]
        cellm = (cell_lab[sl] == cell_id) if (CLIP_TO_CELL and cell_id > 0) else np.ones_like(m)
        others = all_mlo[sl] & ~m
        dist = ndi.distance_transform_edt(~m)
        for ep in ERODE_LEVELS:
            region_name = "Inner" if ep == 0 else f"Inner_erode{ep}"
            m_ep = ndi.binary_erosion(m, iterations=ep) if ep > 0 else m
            rows.append(dict(Image=base, Cell_ID=cell_id, MLO_ID=mlo_id, Region=region_name,
                             Inner_px=0, Outer_px=0, **region_stats(m_ep, c1l, c2l)))
        for name, inner, outer in RINGS:
            band = (dist > inner) & (dist <= outer)
            if CLIP_TO_CELL: band &= cellm
            if EXCLUDE_OTHER_MLOS: band &= ~others
            rows.append(dict(Image=base, Cell_ID=cell_id, MLO_ID=mlo_id, Region=name,
                             Inner_px=inner, Outer_px=outer, **region_stats(band, c1l, c2l)))
    return rows

import time
t = time.time()
rows_test = measure_image(base, c1, c2, lab, keep, all_mlo, cell_lab)
print(f"{len(keep)} MLOs -> {len(rows_test)} rows in {time.time()-t:.2f} s")
pd.DataFrame(rows_test).head(14)

## 15. Run the full batch and save the CSVs
Processes every image in `DIR_MERGE` that has its cell `.zip` and its ILASTIK `.tif`, and saves `MLO_ring_results.csv` and `QC_MLO.csv` in `DIR_OUT`. Also saves one overlay PNG per image in `DIR_OUT/overlays_QC/`.

**Checkpoint:** each image's rows are appended to the CSVs as soon as that image finishes, instead of only at the very end. If the runtime disconnects partway through, just re-run this same cell. It will skip the images already saved and continue from there.

In [ ]:
def overlay_png(base, c1, c2, all_mlo, cell_lab, dir_ov):
    rgb = np.zeros(c1.shape + (3,))
    rgb[..., 0] = normalize_img(c1); rgb[..., 1] = normalize_img(c2)
    rgb[find_boundaries(all_mlo, mode="outer")] = [0, 1, 1]
    rgb[find_boundaries(cell_lab > 0, mode="outer")] = [1, 1, 0]
    plt.figure(figsize=(9, 9)); plt.imshow(rgb); plt.axis("off"); plt.title(base)
    plt.savefig(os.path.join(dir_ov, base + "_QC.png"), dpi=110, bbox_inches="tight"); plt.close()

dir_ov = os.path.join(DIR_OUT, "overlays_QC"); os.makedirs(dir_ov, exist_ok=True)
csv_res_path = os.path.join(DIR_OUT, "MLO_ring_results.csv")
csv_qc_path  = os.path.join(DIR_OUT, "QC_MLO.csv")

RESET = False   # set True to discard any previous partial run and start clean
                # (e.g. after changing the calibration in section 5)
if RESET:
    for fp in (csv_res_path, csv_qc_path):
        if os.path.exists(fp):
            os.remove(fp)
    print("RESET: previous MLO_ring_results.csv / QC_MLO.csv removed.\n")

# Resume support: if this cell already ran partway before (e.g. the runtime
# disconnected), skip images that are already saved in MLO_ring_results.csv
# instead of starting over.
if os.path.exists(csv_res_path):
    done_images = set(pd.read_csv(csv_res_path, usecols=["Image"]).Image.unique())
    print(f"Resuming: {len(done_images)} image(s) already saved, will be skipped.\n")
else:
    done_images = set()

imgs = sorted(glob.glob(os.path.join(DIR_MERGE, "*" + EXT_IMG)))
n_ok = 0
for p in imgs:
    b = os.path.splitext(os.path.basename(p))[0]
    if b in done_images:
        print(f"[skip] {b} (already saved)"); continue
    pc = os.path.join(DIR_CELLS, b + EXT_ROI); ps = os.path.join(DIR_ILASTIK, b + SEG_SUFFIX + EXT_SEG)
    if not (os.path.exists(pc) and os.path.exists(ps)):
        print("[skip]", b, "(missing .zip or .tif)"); continue
    try:
        s = load_segmentation(ps); mv = detect_mlo_label(s) if MLO_LABEL == "auto" else int(MLO_LABEL)
        lb, kp, am = label_mlos(s, mv)
        x1, x2 = load_merge(p); cl, nc = rasterize_cells(pc, x1.shape)
        rows_img = measure_image(b, x1, x2, lb, kp, am, cl)
        qc_img   = qc_single_image(lb, kp, am, x1, x2, cl).assign(Image=b).to_dict("records")
        overlay_png(b, x1, x2, am, cl, dir_ov)

        # Save THIS image's rows right away (append), so progress survives a
        # disconnect -- nothing waits until the whole batch is finished.
        pd.DataFrame(rows_img).to_csv(csv_res_path, mode="a", index=False,
                                       header=not os.path.exists(csv_res_path))
        pd.DataFrame(qc_img).to_csv(csv_qc_path, mode="a", index=False,
                                     header=not os.path.exists(csv_qc_path))
        print(f"[ok] {b}: {len(kp)} MLOs (class {mv})"); n_ok += 1
    except Exception as e:
        print(f"[ERROR] {b}: {e}")

df_res = pd.read_csv(csv_res_path)
df_qc  = pd.read_csv(csv_qc_path)
print(f"\nDone. Images processed this run: {n_ok}")
print("Measurements:", df_res.shape, "| QC:", df_qc.shape)

## 16. Number of MLOs per cell
Counts how many MLOs each cell has (from `df_res`) and saves `MLO_count_per_cell.csv`. If `df_res` isn't in memory (e.g. the runtime restarted after section 15), it's reloaded from the CSV already saved in `DIR_OUT`.

In [ ]:
import os
import pandas as pd

if "df_res" not in globals():
    df_res = pd.read_csv(os.path.join(DIR_OUT, "MLO_ring_results.csv"))
    print("df_res reloaded from MLO_ring_results.csv (it wasn't in memory).")

inner = df_res[df_res.Region == 'Inner']
n_per_cell = (inner.groupby(['Image', 'Cell_ID'])
                      .MLO_ID.nunique()
                      .reset_index(name='n_MLOs')
                      .sort_values(['Image', 'Cell_ID']))

out = os.path.join(DIR_OUT, 'MLO_count_per_cell.csv')
n_per_cell.to_csv(out, index=False)
print(f'Cells: {len(n_per_cell)} | Total MLOs: {n_per_cell.n_MLOs.sum()}')
print('Saved to:', out)
n_per_cell.head(20)

## 17. Inspect one image — all its cells
Loads a (possibly different) image using the shared functions from section 4 and shows all its cells with `view_cell` (section 6).

In [ ]:
IMG_IDX2 = 0     # <-- change this number to move to a different image

imgs2 = sorted(glob.glob(os.path.join(DIR_MERGE, "*" + EXT_IMG)))
print("Images:")
for i, p in enumerate(imgs2):
    print(f"  [{i}] {os.path.splitext(os.path.basename(p))[0]}")

base2   = os.path.splitext(os.path.basename(imgs2[IMG_IDX2]))[0]
p_seg2  = os.path.join(DIR_ILASTIK, base2 + SEG_SUFFIX + EXT_SEG)
p_cell2 = os.path.join(DIR_CELLS, base2 + EXT_ROI)

seg2 = load_segmentation(p_seg2)
mlo_val2 = detect_mlo_label(seg2) if MLO_LABEL == "auto" else int(MLO_LABEL)
lab2, keep2, all_mlo2 = label_mlos(seg2, mlo_val2)
c1_2, c2_2 = load_merge(imgs2[IMG_IDX2])
cell_lab2, n_cells2 = rasterize_cells(p_cell2, c1_2.shape)
mlo_map2 = mlos_by_cell(lab2, keep2, cell_lab2)

print(f"\n>>> [{IMG_IDX2}] {base2} — {n_cells2} cells, {len(keep2)} MLOs\n")
for cid in range(1, cell_lab2.max() + 1):
    view_cell(cid, c1_2, c2_2, lab2, cell_lab2, all_mlo2, mlo_ids=mlo_map2.get(cid, []))

## 18. Log cells to discard
Note down the odd `Cell_ID`s from the image you just looked at (section 17) in `DISCARD`; they accumulate in `discarded_cells.csv`. Repeat by changing `IMG_IDX2` in section 17.

In [ ]:
if "discards" not in globals():
    discards = []

DISCARD = []          # <-- e.g. [3, 7, 12]  (odd Cell_IDs from the image inspected in section 17)
REASON  = "review"    # e.g. "poorly segmented", "2 cells merged", "cut edge"

for cid in DISCARD:
    discards.append(dict(Image=base2, Cell_ID=int(cid), Reason=REASON))

discard_df = (pd.DataFrame(discards).drop_duplicates(["Image", "Cell_ID"])
           if discards else pd.DataFrame(columns=["Image", "Cell_ID", "Reason"]))
path = os.path.join(DIR_OUT, "discarded_cells.csv")
discard_df.to_csv(path, index=False)

print(f"Current image: {base2}  |  flagged now: {DISCARD}")
print(f"Total accumulated: {len(discard_df)} discarded cells across {discard_df.Image.nunique() if len(discard_df) else 0} images")
print("Saved to:", path)
print("\nImages already inspected (with discards):")
print(discard_df.Image.value_counts().to_string() if len(discard_df) else "  (none yet)")

## 19. Apply the discard list to the results
Filters `df_res` by removing the cells listed in `discarded_cells.csv` and saves `MLO_ring_results_filtered.csv` (used from section 22 onward).

In [ ]:
import os
import pandas as pd

if "df_res" not in globals():
    df_res = pd.read_csv(os.path.join(DIR_OUT, "MLO_ring_results.csv"))
    print("df_res reloaded from MLO_ring_results.csv (it wasn't in memory).")

try:
    disc = pd.read_csv(os.path.join(DIR_OUT, "discarded_cells.csv"))
except FileNotFoundError:
    disc = pd.DataFrame(columns=["Image", "Cell_ID"])

df_res["__key"] = df_res.Image.astype(str) + "|" + df_res.Cell_ID.astype(str)
bad = set(disc.Image.astype(str) + "|" + disc.Cell_ID.astype(str))
res_filt = df_res[~df_res["__key"].isin(bad)].drop(columns="__key")

out = os.path.join(DIR_OUT, "MLO_ring_results_filtered.csv")
res_filt.to_csv(out, index=False)
print(f"Original rows: {len(df_res)}  |  after discarding: {len(res_filt)}")
print(f"Cells discarded: {len(bad)}  |  MLOs removed (Inner rows): "
      f"{(df_res['__key'].isin(bad) & (df_res.Region=='Inner')).sum()}")
print("Saved:", out)

## 20. Quick look at the results
Summary to confirm everything ran correctly.

> Requires having run section 15 (creates `df_res` and `df_qc`).

In [ ]:
import os
import pandas as pd

if "df_res" not in globals():
    df_res = pd.read_csv(os.path.join(DIR_OUT, "MLO_ring_results.csv"))
    print("df_res reloaded from MLO_ring_results.csv (it wasn't in memory).")
if "df_qc" not in globals():
    df_qc = pd.read_csv(os.path.join(DIR_OUT, "QC_MLO.csv"))
    print("df_qc reloaded from QC_MLO.csv (it wasn't in memory).")

print("Regions measured:"); print(df_res.Region.value_counts())
print("\nMLOs by QC flag:"); print(df_qc.Flag.value_counts())
print("\nExample rows:")
df_res.head(8)

## 21. Session info (reproducibility)
Prints the Python and package versions used, plus a summary of what was processed.

In [ ]:
import sys, platform
import numpy, scipy, skimage, tifffile, roifile, matplotlib, pandas
print('Python      :', sys.version.split()[0], '|', platform.platform())
for m in [numpy, scipy, skimage, tifffile, roifile, matplotlib, pandas]:
    print(f'{m.__name__:12s}: ' + str(getattr(m, '__version__', '?')))
try:
    print('\nBatch summary:')
    print('  images processed :', df_res.Image.nunique())
    print('  MLOs measured    :', df_res[df_res.Region=="Inner"].shape[0])
    print('  total rows       :', df_res.shape[0])
except NameError:
    print('\n(Run section 15 to see the batch summary.)')

## 22. Plot — inner signal vs erosion (per condition)
One figure per condition. Points are spread horizontally according to the local density of the data (a "sina" plot) so the point cloud takes on a violin-like shape.

Defines the shared `sina_x` helper (density-based jitter), reused by sections 24 and 25.

In [ ]:
import seaborn as sns
from scipy.stats import gaussian_kde

PLOTS_DIR = os.path.join(DIR_OUT, "python_plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

def sina_x(values, width=0.38, seed=0):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    if len(values) < 3 or np.ptp(values) == 0:
        return rng.uniform(-width * 0.15, width * 0.15, size=len(values))
    dens = gaussian_kde(values)(values)
    dens = dens / dens.max()
    return rng.uniform(-1, 1, size=len(values)) * dens * width

ERODE_REGIONS = ["Inner", "Inner_erode1", "Inner_erode2"]
ERODE_LABELS  = {"Inner": "0 px", "Inner_erode1": "1 px", "Inner_erode2": "2 px"}
ERODE_ORDER   = ["0 px", "1 px", "2 px"]

df_erode = df_stats[df_stats.Region.isin(ERODE_REGIONS)].copy()
df_erode["erode_label"] = pd.Categorical(df_erode.Region.map(ERODE_LABELS),
                                          categories=ERODE_ORDER, ordered=True)

for cc in COND_LEVELS:
    d = df_erode[df_erode.condition == cc]
    fig, ax = plt.subplots(figsize=(5, 7))
    for i, lvl in enumerate(ERODE_ORDER):
      yvals = d.loc[d.erode_label == lvl, mk_C1].dropna().values
      print(yvals)
      if len(yvals) == 0:
          continue
      ax.scatter(i + sina_x(yvals), yvals, s=3, alpha=0.5,
                   color=COND_COLORS[cc], linewidths=0)
    means = d.groupby("erode_label", observed=True)[mk_C1].mean()
    counts = d.groupby("erode_label", observed=True).MLO_Global.nunique()
    for i, lvl in enumerate(ERODE_ORDER):
        if lvl in means.index:
            ax.plot([i - 0.3, i + 0.3], [means[lvl]] * 2, color="black", linewidth=2.5)
        ax.text(i, -8, f"n={counts.get(lvl, 0)}", ha="center", va="top", fontsize=9, color="grey")
    ax.set_xticks(range(len(ERODE_ORDER))); ax.set_xticklabels(ERODE_ORDER)
    ax.set_xlim(-0.6, len(ERODE_ORDER) - 0.4)
    ax.set_ylim(0, 150)
    ax.set_xlabel("Erosion level"); ax.set_ylabel("Mean Intensity Inside MLO")
    ax.set_title(cc)
    sns.despine(ax=ax)
    fig.tight_layout()
    fig.savefig(os.path.join(PLOTS_DIR, f"beeswarm_erode_{cc}.png"), dpi=150)
    plt.show()
    print(f"Inner erode plot — {cc} saved.")

## 23. Plot — outer signal vs exclusion distance (per condition)
Exclusion distance 0/2/4 px (fixed 1px ring), one figure per condition. Same style as the previous plot — reuses the `sina_x` helper from section 23, not redefined here.

In [ ]:
EXCL_REGIONS = ["Ring_1px_excl0", "Ring_1px_excl2", "Ring_1px_excl4"]
EXCL_LABELS  = {"Ring_1px_excl0": "0 px", "Ring_1px_excl2": "2 px", "Ring_1px_excl4": "4 px"}
EXCL_ORDER   = ["0 px", "2 px", "4 px"]

df_excl = df_stats[df_stats.Region.isin(EXCL_REGIONS)].copy()
df_excl["excl_label"] = pd.Categorical(df_excl.Region.map(EXCL_LABELS),
                                       categories=EXCL_ORDER, ordered=True)

for cc in COND_LEVELS:
    d = df_excl[df_excl.condition == cc]
    fig, ax = plt.subplots(figsize=(5, 7))
    for i, lvl in enumerate(EXCL_ORDER):
        yvals = d.loc[d.excl_label == lvl, mk_C1].dropna().values
        if len(yvals) == 0:
            continue
        ax.scatter(i + sina_x(yvals), yvals, s=3, alpha=0.5,
                   color=COND_COLORS[cc], linewidths=0)
    means = d.groupby("excl_label", observed=True)[mk_C1].mean()
    counts = d.groupby("excl_label", observed=True).MLO_Global.nunique()
    for i, lvl in enumerate(EXCL_ORDER):
        if lvl in means.index:
            ax.plot([i - 0.3, i + 0.3], [means[lvl]] * 2, color="black", linewidth=2.5)
        ax.text(i, -8, f"n={counts.get(lvl, 0)}", ha="center", va="top", fontsize=9, color="grey")
    ax.set_xticks(range(len(EXCL_ORDER))); ax.set_xticklabels(EXCL_ORDER)
    ax.set_xlim(-0.6, len(EXCL_ORDER) - 0.4)
    ax.set_ylim(0, 150)
    ax.set_xlabel("Exclusion distance before ring"); ax.set_ylabel("Mean Intensity Outside MLO")
    ax.set_title(cc)
    sns.despine(ax=ax)
    fig.tight_layout()
    fig.savefig(os.path.join(PLOTS_DIR, f"beeswarm_excl_{cc}.png"), dpi=150)
    plt.show()
    print(f"Outer exclusion-distance plot — {cc} saved.")

### 24. Compute the condensed/soluble ratio per MLO (**EDIT**)
Uses `res_filt` (section 19) if available, otherwise reloads `MLO_ring_results_filtered.csv`. Applies the per-MLO intensity filter (`Max_<READOUT> < INTENS_MAX`, section 3) and computes the condensed/soluble ratio at the selected erosion and dilation values for every MLO, tagging each with its condition.

In [ ]:
if "res_filt" in globals():
    df_stats = res_filt.copy()
else:
    df_stats = pd.read_csv(os.path.join(DIR_OUT, "MLO_ring_results_filtered.csv"))

df_stats["MLO_Global"] = (df_stats.Image.astype(str) + "|" +
                          df_stats.Cell_ID.astype(str) + "|" + df_stats.MLO_ID.astype(str))
df_stats["condition"]  = pd.Categorical(df_stats.Image.astype(str).apply(condition_of),
                                        categories=COND_LEVELS)

mk_max = f"Max_{READOUT}"
mk_C1  = f"Mean_{READOUT}"

# Zero-area regions (e.g. an MLO fully consumed by erosion) were saved as the
# string "NA" instead of a number -- coerce to numeric (NA -> NaN) before any
# numeric operation, otherwise groupby/max errors on a mixed str/float column.
for col in (mk_max, mk_C1):
    df_stats[col] = pd.to_numeric(df_stats[col], errors="coerce")

# Per-MLO intensity filter: discard the MLO if Max_<READOUT> reaches INTENS_MAX in ANY region
mlo_max  = df_stats.groupby("MLO_Global")[mk_max].max()
ok_mlos  = mlo_max[mlo_max < INTENS_MAX].index
n_before = mlo_max.shape[0]
df_stats = df_stats[df_stats.MLO_Global.isin(ok_mlos)]
print(f"Filter {mk_max} < {INTENS_MAX}: {n_before} -> {len(ok_mlos)} MLOs "
      f"(discarded {n_before - len(ok_mlos)}, {100*(n_before-len(ok_mlos))/n_before:.1f}%)")

# --- EDIT: Set erosion and dilation values to compute condensed/soluble ratio ---
ERODE_REGION = "Inner_erode2"
RING_REGION  = "Ring_1px_excl4"
assert ERODE_REGION in df_stats.Region.unique(), f"{ERODE_REGION} not found — check calibration (section 5)."
assert RING_REGION  in df_stats.Region.unique(), f"{RING_REGION} not found — check calibration (section 5)."

inside  = df_stats[df_stats.Region == ERODE_REGION][["MLO_Global", "condition", mk_C1]].rename(columns={mk_C1: "inside"})
outside = df_stats[df_stats.Region == RING_REGION][["MLO_Global", mk_C1]].rename(columns={mk_C1: "outside"})
df_ratio = inside.merge(outside, on="MLO_Global")
df_ratio = df_ratio[(df_ratio.inside > 0) & (df_ratio.outside > 0)].copy()
df_ratio["ratio"] = df_ratio.inside / df_ratio.outside

print(f"\nMLOs with a valid ratio: {len(df_ratio)}")
print(df_ratio.groupby("condition", observed=True).agg(
    n=("MLO_Global", "nunique"), median_ratio=("ratio", "median")))

## 25. Plot — inside/outside ratio (all conditions)
One figure with the three conditions side by side, using `df_ratio` from section 24.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 7))
for i, cc in enumerate(COND_LEVELS):
    yvals = df_ratio.loc[df_ratio.condition == cc, "ratio"].dropna().values
    if len(yvals) == 0:
        continue
    ax.scatter(i + sina_x(yvals), yvals, s=3, alpha=0.5, color=COND_COLORS[cc], linewidths=0)
means  = df_ratio.groupby("condition", observed=True).ratio.mean()
counts = df_ratio.groupby("condition", observed=True).MLO_Global.nunique()
for i, cc in enumerate(COND_LEVELS):
    if cc in means.index:
        ax.plot([i - 0.3, i + 0.3], [means[cc]] * 2, color="black", linewidth=2.5)
    ax.text(i, -2, f"n={counts.get(cc, 0)}", ha="center", va="top", fontsize=9, color="grey")
ax.axhline(1, linestyle="--", color="grey", linewidth=0.9)
ax.set_xticks(range(len(COND_LEVELS))); ax.set_xticklabels(COND_LEVELS)
ax.set_xlim(-0.6, len(COND_LEVELS) - 0.4)
ax.set_ylim(0, 50)
ax.set_xlabel("Condition"); ax.set_ylabel("Inside / Outside ratio")
ax.set_title("Inside/Outside ratio per MLO\nerode 2px (1/2r) / exclusion 4px (1r)")
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "ratio_erode2px_excl4px_allconditions.png"), dpi=150)
plt.show()
print("Inside/outside ratio plot saved.")

## 26. Plot — inner vs outer signal, scatter (all conditions)
Condensed (erode 2px) vs soluble (exclusion 4px), colored by condition, using `df_ratio` from section 24.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for cc in COND_LEVELS:
    d = df_ratio[df_ratio.condition == cc]
    ax.scatter(d.outside, d.inside, s=6, alpha=0.35, color=COND_COLORS[cc], label=cc)
ax.plot([0, 150], [0, 150], linestyle="--", color="grey", linewidth=0.9)
ax.set_xlim(0, 150); ax.set_ylim(0, 150)
ax.set_xlabel("Mean Intensity Outside MLO (Ring 1px, exclusion 4px)")
ax.set_ylabel("Mean Intensity Inside MLO (erode 2px)")
ax.set_title(f"Inner vs outer signal — all conditions\nfilter {mk_max} < {INTENS_MAX}")
ax.legend(title=None, loc="upper left", frameon=False)
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "scatter_erode2px_excl4px_allconditions.png"), dpi=150)
plt.show()
print("Scatter plot saved. All Python plots are in:", PLOTS_DIR)